<a href="https://colab.research.google.com/github/samueltsaii/Personal_Projects/blob/main/Semantic_Liver_Mass_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The goal of the project is to create a model that creates a segmentation around a given liver mass on an ultrasound image and classifies whether the mass is malignant or not. Annotations for the data set was generously curated by Xu et al(2022) under creative commons.



Xu Yiming, Zheng Bowen, Liu Xiaohong, Wu Tao, Ju Jinxiu, Wang Shijie, Lian Yufan, Zhang Hongjun, Liang Tong, Sang Ye, Jiang Rui, Wang Guangyu, Ren Jie, & Chen Ting. (2022). Annotated Ultrasound Liver images [Data set]. Zenodo. https://doi.org/10.5281/zenodo.7272660

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import json

Loading the first image in a class from a data set to test image path:

In [ ]:
img = cv2.imread("drive/MyDrive/data/Benign/Benign/image/1.jpg")
if img is None:
    print("Error: Image not found")
else:
    img_0 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_0)
    plt.title("First image from data set")
    plt.axis("off")
    plt.show()


Seperating data accoording to class:

In [ ]:
base_dir = "drive/MyDrive/data"

categories = ["Benign", "Malignant"]

data = []

for category in categories:
    image_dir = os.path.join(base_dir, category, category, "image")
    seg_dir = os.path.join(base_dir, category, category, "segmentation", "mass")

    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(("png", "jpg", "jpeg"))])
    json_files = sorted([f for f in os.listdir(seg_dir) if f.lower().endswith("json")])

    for img, jsn in zip(image_files, json_files):
        data.append({
            "Category":category,
            "Image_Path":os.path.join(image_dir, img),
            "Segmentation_JSON_Path": os.path.join(seg_dir, jsn)
        })

df = pd.DataFrame(data)

In [ ]:
df.head()

In [ ]:
df.loc[0]["Segmentation_JSON_Path"]

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

Getting the frequency count per class.

Note: There is a disproprtionatly greater number of images labelled "malignant". Sensitivity for detecting malignancy in the mass is prioritized in this context to increase patient survival.  

In [ ]:
counts = df['Category'].value_counts()

categories = counts.index
frequencies = counts.values

plt.figure(figsize=(8, 5))

colors = ['red' if cat == 'Malignant' else 'green' for cat in categories]

plt.bar(categories, frequencies, color=colors)

plt.xlabel("Mass Categories")

plt.ylabel("Frequency")
plt.title("Frequency per Mass Categories")

plt.show()

Creating the mask from provided JSON files:

In [ ]:
def generate_mask_from_json(json_path, image_shape):
    with open(json_path, 'r') as f:
        points_list = json.load(f)

    mask = np.zeros(image_shape[:2], dtype=np.uint8)

    points = np.array(points_list, dtype=np.int32)

    if len(points) >= 3:
        cv2.fillPoly(mask, [points], 255)

    return mask

def display_images_with_masks(df):
    categories = df['Category'].unique()
    fig, axes = plt.subplots(len(categories), 5, figsize=(20, 10))
    fig.suptitle("5 Randomly Drawn Liver Image Samples.\n {Green: Benign, Red: Malignant}", fontsize=20)

    for row_idx, category in enumerate(categories):

        samples = df[df['Category'] == category].sample(5, random_state=42).reset_index(drop=True)

        for col_idx, row in samples.iterrows():
            img = cv2.imread(row['Image_Path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            try:
                mask = generate_mask_from_json(row['Segmentation_JSON_Path'], img.shape)
            except Exception as e:
                print(f"Error reading mask for {row['Segmentation_JSON_Path']}: {e}")
                mask = np.zeros(img.shape[:2], dtype=np.uint8)

            overlay = img.copy()
            alpha = 0.65
            beta = 0.35
            colour = [255, 0, 0] if 'Malignant' in row["Category"] else [0, 255, 0]
            overlay[mask == 255] = colour
            transparent_image = cv2.addWeighted(img, alpha, overlay, beta, 0)

            axes[row_idx, col_idx].imshow(transparent_image)
            axes[row_idx, col_idx].axis('off')

            if col_idx == 0:
                axes[row_idx, col_idx].set_ylabel(category, fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

display_images_with_masks(df)

Modifying the data set to be more balanced:

In [ ]:
all_malignant_samples = df[df['Category'] == 'Malignant']
sampled_malignant = all_malignant_samples.sample(n = 200)

all_benign_samples = df[df['Category'] == 'Benign']

df = pd.concat([sampled_malignant, all_benign_samples]).reset_index(drop=True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.v2 as transforms
from PIL import Image

The previous iteration attempted to follow the original U-NET architectures. Given limited computed resources provided by the Collab enviroment and slow convergence, the architecture was modified to have batch normalization and smaller channel sizes (for computation purposes).

In [ ]:
class Unet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.e1 = conv_block(3, 16)
        self.e2 = conv_block(16, 32)
        self.e3 = conv_block(32, 64)
        self.e4 = conv_block(64, 128)
        self.e5 = conv_block(128, 256)

        self.pool = nn.MaxPool2d(2, 2)

        # Decoder
        self.up4 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.d4 = conv_block(128, 64)

        self.up3 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d3 = conv_block(64, 32)

        self.up2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d2 = conv_block(64, 32)

        self.up1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.d1 = conv_block(32, 16)

        self.outconv = nn.Conv2d(16, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        c1 = self.e1(x)
        p1 = self.pool(c1)

        c2 = self.e2(p1)
        p2 = self.pool(c2)

        c3 = self.e3(p2)
        p3 = self.pool(c3)

        c4 = self.e4(p3)
        p4 = self.pool(c4)

        c5 = self.e5(p4)

        # Decoder
        u4 = self.up4(c5)
        u4 = torch.cat([u4, c4], dim=1)
        d4 = self.d4(u4)

        u3 = self.up3(d4)
        u3 = torch.cat([u3, c3], dim=1)
        d3 = self.d3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, c2], dim=1)
        d2 = self.d2(u2)

        u1 = self.up1(d2)
        u1 = torch.cat([u1, c1], dim=1)
        d1 = self.d1(u1)

        return self.outconv(d1)

__getitem__ has been modified to return tensors of binary values to accomodate for dice metrics

In [ ]:
class LiverSegmentationDataset(Dataset):
    def __init__(self, dataframe, image_key, annotation_key, transform = None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images = dataframe[image_key]
        self.annotations = dataframe[annotation_key]
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = generate_mask_from_json(self.annotations[idx], image.shape)

        image = Image.fromarray(image)
        mask = Image.fromarray(mask)

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0).float()

        return image, mask

In [ ]:
# Instantiate dice coefficients
def dice_coef(preds, targets, smooth=1e-6):
    preds = torch.argmax(preds, dim=1)
    preds = preds.view(-1)
    targets = targets.view(-1)

    intersection = (preds * targets).sum()
    dice = (2.0 * intersection + smooth) / (preds.sum() + targets.sum() + smooth)
    return dice

Ultrasound data is inherently "noisey" by nature. Thus, a soft dice loss is used.

In [ ]:
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(SoftDiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)

        probs_liver = probs[:, 1, :, :]

        probs_flat = probs_liver.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1).float()

        intersection = (probs_flat * targets_flat).sum()

        dice_score = (2. * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)

        return 1 - dice_score

The main difficulty encountered during training was balancing computation and resolution. Sizes 128 x 128 and 256 x 256 were tried. Since masses are a subset of the liver image and can be very small, information loss typically occurs when an image goes through convolution. On the otherhand, if the channels and image size were made too big, the GPU storage space would be exceeded during the first pass!

In [ ]:
transform = transforms.Compose([
    transforms.Resize((512,512)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(contrast=0.2),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
])

data_set = LiverSegmentationDataset(df, "Image_Path", "Segmentation_JSON_Path", transform)
img, mask = data_set[0]
print(f"Mask unique values: {torch.unique(mask)}")

In [ ]:
train_size = int(0.8 * len(data_set))
valid_size = len(data_set) - train_size

train_data_set, valid_data_set = random_split(data_set, [train_size, valid_size])

In [ ]:
train_loader = DataLoader(train_data_set, batch_size = 32, shuffle = True)
valid_loader = DataLoader(valid_data_set, batch_size = 32, shuffle = False)

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        inputs = F.softmax(inputs, dim=1)
        inputs = inputs[:, 1, :, :].reshape(-1)
        targets = targets.reshape(-1).float()

        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)
        BCE = F.binary_cross_entropy(inputs, targets, reduction='mean')

        return BCE + dice_loss

In [ ]:
#Instantiating hyperparameters
model = Unet(n_classes = 2)
Epochs = 100
learning_rate = 1e-6
optimizer = optim.Adam(model.parameters(), lr= learning_rate, weight_decay=1e-5)
criterion = SoftDiceLoss()
es = EarlyStopping()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def get_hard_dice_score(preds, targets):
    return dice_coef(preds, targets, smooth=1e-6)

model.to(device)

train_losses, valid_losses = [], []
valid_dice_scores = []

for Epoch in range(Epochs):
    model.train()
    train_running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.squeeze(1).to(device=device, dtype=torch.long)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_running_loss += loss.item() * images.size(0)

    epoch_train_loss = train_running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    valid_running_loss = 0.0
    valid_running_dice = 0.0

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.squeeze(1).to(device=device, dtype=torch.long)

            outputs = model(images)

            loss = criterion(outputs, labels)
            valid_running_loss += loss.item() * images.size(0)

            batch_dice = get_hard_dice_score(outputs, labels)
            valid_running_dice += batch_dice.item() * images.size(0)

    epoch_valid_loss = valid_running_loss / len(valid_loader.dataset)
    epoch_valid_dice = valid_running_dice / len(valid_loader.dataset)

    valid_losses.append(epoch_valid_loss)
    valid_dice_scores.append(epoch_valid_dice)


    print(f'Epoch {Epoch + 1}/{Epochs}')
    print(f'Train Loss (Soft): {epoch_train_loss:.4f} | Valid Loss (Soft): {epoch_valid_loss:.4f}')
    print(f'Valid Dice (Hard): {epoch_valid_dice:.4f}')
    print('-' * 30)


Visualization of the masks produced by the model and ground truths.

In [ ]:
def visualize_results(model, dataloader, device, num_samples=3):
    model.eval()
    images, labels = next(iter(dataloader))

    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    plt.figure(figsize=(12, 4 * num_samples))

    for i in range(num_samples):
        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(images[i].cpu().permute(1, 2, 0))
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(labels[i].cpu().squeeze(), cmap='gray')
        plt.title("Ground Truth Mask")
        plt.axis('off')

        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(preds[i].cpu(), cmap='Purples', alpha=0.5)
        plt.title("Predicted Mask")
        plt.axis('off')

    plt.tight_layout()
    plt.savefig('sample_predictions.png')

visualize_results(model, valid_loader, device)